# 04 — Inspect analysis tables

This notebook follows the output-checking pattern in `Raster_processing.ipynb`: identify the concrete Parquet file written by the pipeline, query a small preview with DuckDB, and only then move into a larger analysis.

## 1. Point to one completed flightline

Keep the path tied to the same `base_folder` and flightline identifier used by the pipeline notebook. `RUN = False` inventories files without opening a table.

In [ ]:
from pathlib import Path

import duckdb

RUN = False
base_folder = Path("outputs/neon_notebook")
flight_stem = "NEON_D13_NIWO_DP1_L019-1_20230815_directional_reflectance"
flight_dir = base_folder / flight_stem

## 2. Inventory the tables

Merged files are usually the analysis entry point; the other Parquet files are useful for tracing an individual sensor product or debugging a merge.

In [ ]:
parquet_paths = sorted(flight_dir.glob("*.parquet")) if flight_dir.exists() else []
print(f"Found {len(parquet_paths)} Parquet files in {flight_dir}")
for path in parquet_paths:
    print(f"  {path.name}: {path.stat().st_size:,} bytes")

merged_candidates = [path for path in parquet_paths if "merged" in path.name]
parquet_path = merged_candidates[0] if merged_candidates else (parquet_paths[0] if parquet_paths else None)
print(f"Preview target: {parquet_path}")

## 3. Query a small preview

DuckDB reads the Parquet file directly, so this preview does not load the full table into memory. Increase the limit only after checking its row and column counts.

In [ ]:
if RUN and parquet_path is not None:
    connection = duckdb.connect()
    table_summary = connection.execute(
        "SELECT count(*) AS rows FROM read_parquet(?)",
        [str(parquet_path)],
    ).fetchdf()
    table_preview = connection.execute(
        "SELECT * FROM read_parquet(?) LIMIT 5",
        [str(parquet_path)],
    ).fetchdf()
    display(table_summary)
    display(table_preview)
    print(f"Columns ({len(table_preview.columns)}): {list(table_preview.columns)}")
elif parquet_path is None:
    print("No Parquet output is available yet. Run or resume notebook 00.")
else:
    print("Inventory complete. Set RUN = True to query the selected table.")

## 4. Interpret the preview

Check identifiers and coordinate columns before spectral values. Confirm that the selected file represents the expected flightline and sensor products, then use notebook 05 to compare these tabular outputs with spatial QA.